In [1]:
import numpy as np
import yamnet as yamnet_model
import params as yamnet_params

import soundfile as sf
import os
import resampy
import pandas as pd
import ast
import tqdm

import tensorflow as tf
import tensorboard as tb
from tensorboard.plugins import projector

# Load the YAMNet model
params = yamnet_params.Params()
yamnet = yamnet_model.yamnet_frames_model(params)
yamnet.load_weights('yamnet.h5')
yamnet_classes = yamnet_model.class_names('yamnet_class_map.csv')

In [2]:
def load_and_preprocess_audio(file_path, target_sr=16000):
    # Load audio file
    audio, sr = sf.read(file_path)
    # If stereo, take the mean to convert to mono
    if len(audio.shape) > 1:
        audio = np.mean(audio, axis=1)
    # Resample to 16kHz if necessary
    if sr != target_sr:
        audio = resampy.resample(audio, sr, target_sr)
    return audio

def extract_embeddings(yamnet_model, audio_data, params):
    # Predict using YAMNet
    scores, embeddings, spectrogram = yamnet_model(audio_data)
    return embeddings.numpy()

def clean_df(df):
    df = df.dropna()
    df = df.drop_duplicates()
    
    # add audio file path to each row
    df['path'] = df['files'].apply(lambda name: os.path.join(audio_folder, name))
    return df

def clean_class(class_string):
    try:
        # Safely convert the string representation of the list back to a list
        class_list = ast.literal_eval(class_string)
        # Return the first element if the list is not empty
        if class_list:  # Checks if the list is not empty
            return class_list[0]  # Returns the first element of the list
        else:
            return None  # Or some default value or empty string ''
    except:
        return None  # In case of any exception, return None or some default value

In [3]:
# where the audio files are stored
audio_folder = r'\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT\20231211_SANTUR\3-Medidas\P2-CONTENEDORES\AUDIOMOTH'
# where the predictions are stored
csv_file_path = r'\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT\20231211_SANTUR\5-Resultados\P2_CONTENEDORES\URBAN_Model\Predictions\Urban_Model_P2_CONTENEDORES_v1_0.csv'

df = pd.read_csv(csv_file_path)
df = clean_df(df)

# get the first class in the list of classes_original column
df['label'] = df['classes_original'].apply(clean_class)
df

,files,datetime,classes_custom,probabilities_custom,sum_probs_custom,classes_original,probabilities_original,sum_probs_original,path,label
0,20231211_144935.WAV,2023-12-11 14:49:35,"['Motorised transport', 'Geonature', 'Other So...",[0.10812548 0.10812548 0.06801586],0.284267,"['Vehicle', 'Wind', 'Wind noise (microphone)']","[0.3698452, 0.068612404, 0.06495217]",0.503410,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,Vehicle
1,20231211_150440.WAV,2023-12-11 15:04:40,"['Motorised transport', 'Geonature', 'Other So...",[0.11777134 0.11777134 0.06831889],0.303862,"['Vehicle', 'Wind', 'Boat, Water vehicle']","[0.3180183, 0.07247015, 0.068541884]",0.459030,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,Vehicle
2,20231211_151945.WAV,2023-12-11 15:19:45,"['Motorised transport', 'Other Sounds', 'Backg...",[0.09061588 0.09061588 0.07942747],0.260659,"['Vehicle', 'Speech', 'Boat, Water vehicle']","[0.22406884, 0.07079232, 0.044671834]",0.339533,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,Vehicle
3,20231211_153450.WAV,2023-12-11 15:34:50,"['Motorised transport', 'Background', 'Geonatu...",[0.08266572 0.08266572 0.07569595],0.241027,"['Vehicle', 'Aircraft', 'Speech']","[0.25925973, 0.055592418, 0.05295557]",0.367808,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,Vehicle
4,20231211_154955.WAV,2023-12-11 15:49:55,"['Motorised transport', 'Other Sounds', 'Geona...",[0.09267476 0.09267476 0.09224785],0.277597,"['Vehicle', 'Speech', 'Boat, Water vehicle']","[0.20645754, 0.07090294, 0.050905224]",0.328266,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,Vehicle
...,...,...,...,...,...,...,...,...,...,...
754,20231219_200620.WAV,2023-12-19 20:06:20,"['Motorised transport', 'Other Sounds', 'Elect...",[0.1471351 0.1471351 0.10910264],0.403373,"['Vehicle', 'Motor vehicle (road)', 'Car']","[0.27237082, 0.076590285, 0.061655372]",0.410616,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,Vehicle
755,20231219_203630.WAV,2023-12-19 20:36:30,"['Animal', 'Other human', 'Motorised transport']",[0.17106418 0.17106418 0.14460346],0.486732,"['Snort', 'Breathing', 'Animal']","[0.09449526, 0.09426089, 0.08509293]",0.273849,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,Snort
756,20231219_213650.WAV,2023-12-19 21:36:50,"['Animal', 'Motorised transport', 'Other Sounds']",[0.22350836 0.22350836 0.10826856],0.555285,"['Vehicle', 'Animal', 'Outside, rural or natur...","[0.10939811, 0.06528124, 0.057132915]",0.231812,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,Vehicle
757,20231220_024535.WAV,2023-12-20 02:45:35,"['Motorised transport', 'Human movement', 'Oth...",[0.20533019 0.20533019 0.14912482],0.559785,"['Vehicle', 'Bicycle', 'Boat, Water vehicle']","[0.34699473, 0.33702028, 0.08668762]",0.770703,\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...,Vehicle


In [4]:
print(f"These are the labels:\n\n{df['label'].unique()}")
print(f"\nThere are {len(df['label'].unique())} labels")

These are the labels:

['Vehicle' 'Bicycle' 'Animal' 'Inside, small room' 'Breathing'
 'White noise' 'Heart sounds, heartbeat' 'Snort']

There are 8 labels


In [6]:
embeddings = []
file_names = []
labels = [] 

for index, row in df.iterrows():
    file_name = row['files']
    audio_path = row['path'] 
    label = row['label']
    print(f'Processing {file_name}')
    
    audio_data = load_and_preprocess_audio(audio_path)
    
    # audio data shape matches what YAMNet expects
    audio_data = audio_data.reshape(-1, 1)
    
    # extract embeddings
    print('Extracting embeddings...')
    embedding = extract_embeddings(yamnet, audio_data, params)
    
    embeddings.append(embedding.mean(axis=0))
    print(f"embeddings shape: {embedding.shape}")
    
    file_names.append(file_name)
    labels.append(label)
    
    print(f'Finished processing {file_name}\n')

Processing 20231211_144935.WAV
Extracting embeddings...
embeddings shape: (1874, 1024)
Finished processing 20231211_144935.WAV

Processing 20231211_150440.WAV
Extracting embeddings...
embeddings shape: (1874, 1024)
Finished processing 20231211_150440.WAV

Processing 20231211_151945.WAV
Extracting embeddings...
embeddings shape: (1874, 1024)
Finished processing 20231211_151945.WAV

Processing 20231211_153450.WAV
Extracting embeddings...
embeddings shape: (1874, 1024)
Finished processing 20231211_153450.WAV

Processing 20231211_154955.WAV
Extracting embeddings...
embeddings shape: (1874, 1024)
Finished processing 20231211_154955.WAV

Processing 20231211_160500.WAV
Extracting embeddings...
embeddings shape: (1874, 1024)
Finished processing 20231211_160500.WAV

Processing 20231211_162005.WAV
Extracting embeddings...
embeddings shape: (1874, 1024)
Finished processing 20231211_162005.WAV

Processing 20231211_163510.WAV
Extracting embeddings...
embeddings shape: (1874, 1024)
Finished processi

In [ ]:
# logs directory
log_dir = 'logs/embeddings'
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

# metadata
metadata_path = os.path.join(log_dir, 'metadata.tsv')
with open(metadata_path, 'w') as metadata_file:
    metadata_file.write("Filename\tLabel\n")  # Header
    for file_name, label in zip(file_names, labels):
        metadata_file.write(f"{file_name}\t{label}\n")
        
# save embeddings
embeddings_tensor = tf.Variable(embeddings, name='yamnet_embeddings')
checkpoint = tf.train.Checkpoint(embedding=embeddings_tensor)
checkpoint.save(os.path.join(log_dir, 'embedding.ckpt'))

# setup projector config
config = projector.ProjectorConfig()
embedding = config.embeddings.add()

# link embeddings to their metadata file
embedding.tensor_name = 'embedding/.ATTRIBUTES/VARIABLE_VALUE'
embedding.metadata_path = 'metadata.tsv'
projector.visualize_embeddings(log_dir, config)

# warning
cwd = os.getcwd()

print(f"Go to the directory containing the logs folder {cwd} and run the following command")
print(f'Run `tensorboard --logdir={log_dir}` to visualize embeddings in TensorBoard.')

Go to the directory containing the logs folder c:\Users\GIS2\Documents\santi\GitHub\AAC\AI_Model\Port_Model\Visualization\tensorboar_embeddings and run the following command
Run `tensorboard --logdir=logs/embeddings` to visualize embeddings in TensorBoard.
